In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()


In [ ]:
import scanpy as sc
import os
import json
import math
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix
from collections import Counter,defaultdict
import scanpy.external as sce
sc.settings.verbosity = 3
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
def plot_custom_umap(adata, obs_key, save_dir, output_name, color_mapping_json=None, species="all"):
    """
    Draw a UMAP and save the legend separately as a PDF.
    
    Parameters:
    - adata: AnnData object
    - obs_key: str, column name in adata.obs (e.g. 'cell_type', 'tissue')
    - save_dir: str, output directory for the figures
    - output_name: str, prefix of the output filenames
    - color_mapping_json: str, path to a JSON colour map (optional; scanpy defaults are used if absent)
    - species: str, species name used in filenames (default "all")
    """
    
    # Ensure the output directory exists
    os.makedirs(save_dir, exist_ok=True)
    
    # Ensure obs_key is categorical; otherwise .cat.categories raises
    if not hasattr(adata.obs[obs_key], 'cat'):
        adata.obs[obs_key] = adata.obs[obs_key].astype('category')
        
    cats = adata.obs[obs_key].cat.categories

    # 1. Build the colour mapping
    if color_mapping_json and os.path.exists(color_mapping_json):
        with open(color_mapping_json, "r") as f:
            color_mapping_dict = json.load(f)
        adata.uns[f"{obs_key}_colors"] = [color_mapping_dict.get(c, "#bdbdbd") for c in cats]
    else:
        # If no JSON was supplied or the file is missing, let scanpy assign colours first
        if f"{obs_key}_colors" not in adata.uns:
            n = adata.obs[obs_key].nunique()
            colors = sns.color_palette("husl", n)
            sc.pl.umap(adata, color=obs_key, show=False,palette=colors) 
            plt.close()

    colors = adata.uns[f"{obs_key}_colors"]

    # 2. Draw and save the main UMAP (no axes, legend or title)
    fig_umap = plt.figure(figsize=(6, 6))
    ax_umap = fig_umap.add_axes([0.02, 0.02, 0.96, 0.96])

    sc.pl.umap(
        adata,
        color=obs_key,
        show=False,
        frameon=False,
        ax=ax_umap,
        legend_loc=None,
        size=120000 / adata.n_obs * 2.5,
        title=''
    )

    ax_umap.set_aspect("equal")
    ax_umap.set_axis_off()

    umap_filename = f"{save_dir}/{output_name}_{species}_{obs_key}_umap.png"
    fig_umap.savefig(umap_filename, dpi=300, format="png")
    plt.close(fig_umap)

    # 3. Draw and save the legend separately
    cat2num = {cat: i+1 for i, cat in enumerate(cats)}

    legend_elements = [
        Line2D([0], [0], marker='o', color='w',
#                label=f"{cat2num[cat]:02d} {cat}",
               label=f"{cat}",
               markerfacecolor=col, markersize=12)
        for cat, col in zip(cats, colors)
    ]

    items_per_column = 20
    num_columns = math.ceil(len(legend_elements) / items_per_column)

    # Compute the legend canvas size dynamically
    fig_width = 3 * num_columns
    fig_height = max(4, 0.4 * items_per_column)
    
    fig_legend, axes = plt.subplots(1, num_columns, figsize=(fig_width, fig_height))

    if num_columns == 1:
        axes = [axes]

    for col_idx in range(num_columns):
        start = col_idx * items_per_column
        end = start + items_per_column
        subset = legend_elements[start:end]

        ax_col = axes[col_idx]
        ax_col.axis("off")

        if subset:  # Ensure the subset is not empty
            ax_col.legend(
                handles=subset,
                frameon=False,
                loc='upper left',
                fontsize=10,
                borderpad=0.2,
                handletextpad=0.3
            )

    plt.tight_layout()
    legend_filename = f"{save_dir}/{output_name}_{species}_{obs_key}_umap_legend.pdf"
    fig_legend.savefig(legend_filename, bbox_inches="tight", dpi=300)
    plt.close(fig_legend)


In [ ]:
## Confusion matrix
def plot_cm_from_df(df, true_col, pred_col, save_name, species=None, threshold=0.10, onlyTrueLabels=True):
    """
    Draw a cell-type confusion matrix from a DataFrame.
    """
    # If a species name is given, append it to the output filename
    if species is not None:
        save_name = f"{save_name}_{species}"

    # 1. Extract the true and predicted labels
    cell_types_labels = df[true_col].values
    cell_types_predictions = df[pred_col].values

    # 2. Collect all unique labels (union of true and predicted)
    all_labels = np.union1d(np.unique(cell_types_labels), np.unique(cell_types_predictions))

    # 3. Compute the confusion matrix
    cm = confusion_matrix(cell_types_labels, cell_types_predictions, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=all_labels, columns=all_labels)
    cm_np = cm_df.to_numpy().astype('float')

    # 4. Row-normalise to get accuracy/proportions (guarding against division by zero)
    row_sums = cm_np.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  
    accuracy_matrix = cm_np / row_sums

    accuracy_df = pd.DataFrame(accuracy_matrix, index=all_labels, columns=all_labels)

    # 5. Choose the true labels to display (rows)
    if onlyTrueLabels:
        true_labels_unique = np.unique(cell_types_labels)
    else:
        true_labels_unique = all_labels
        
    # Reorder all columns: true labels first, then the rest
    remaining_labels = [label for label in all_labels if label not in true_labels_unique]
    ordered_labels = list(true_labels_unique) + remaining_labels
    accuracy_df = accuracy_df[ordered_labels]

    # 6. Filter columns by threshold
    filtered_df = accuracy_df.loc[:, (accuracy_df > threshold).any(axis=0)]

    # 7. Move the largest predicted probability close to the diagonal
    for i, label in enumerate(true_labels_unique):
        if label in filtered_df.index:
            max_col = filtered_df.loc[label].idxmax()
            if max_col in filtered_df.columns:
                max_idx = filtered_df.columns.get_loc(max_col)
                if max_idx > i and i < len(filtered_df.columns):
                    cols = list(filtered_df.columns)
                    cols[i], cols[max_idx] = cols[max_idx], cols[i]
                    filtered_df = filtered_df[cols]

    # 8. Compute the figure size dynamically
    label_count_x = len(filtered_df.columns)
    label_count_y = len(true_labels_unique)
    figsize = (max(5, label_count_x * 0.5), max(4, label_count_y * 0.5))

    # 9. Plot
    plt.figure(figsize=figsize)
    sns.heatmap(
        filtered_df.loc[true_labels_unique], 
        annot=True, 
        fmt='.2f', 
        cmap='Blues', 
        cbar=False, 
        annot_kws={"size": 10}
    )

    plt.xlabel('Predicted Cell Type', fontsize=12)
    plt.ylabel('Annotated Cell Type', fontsize=12)

    plt.xticks(np.arange(len(filtered_df.columns)) + 0.5, filtered_df.columns, rotation=45, ha='right')
    plt.yticks(np.arange(len(true_labels_unique)) + 0.5, true_labels_unique, rotation=0)

    # 10. Save and show
    plt.savefig(f"{save_name}.pdf", dpi=300, format='pdf', bbox_inches='tight')
    plt.show()


In [ ]:
def plot_sankey_from_adata(
    adata,
    source_col,
    target_col,
    title="Sankey Diagram",
    source_colors=None,
    target_color="#f0f0f0",
    width=1000,
    height=800,
    interactive=False,   # Static by default
    output_file=None     # Optional: save to file
):
    import plotly.graph_objects as go
    import plotly.io as pio
    """
    Draw a Sankey diagram from two columns of AnnData.obs (loop-free).
    """

    # 1. Prepare the data
    df = adata.obs[[source_col, target_col]].dropna()
    count_matrix = pd.crosstab(df[source_col], df[target_col], margins=False)

    # 2. Extract nodes and separate the source/target namespaces
    source_categories = count_matrix.index.tolist()
    target_categories = count_matrix.columns.tolist()

    # Suffix target nodes so they cannot clash with source names
    target_categories_renamed = [f"{cat} " for cat in target_categories]

    all_categories = source_categories + target_categories_renamed
    node_indices = {category: i for i, category in enumerate(all_categories)}

    # 3. Source-node colours
    if source_colors is None:
        source_colors = [
            '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
            '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
            '#aec7e8', '#ffbb78'
        ]
    num_sources = len(source_categories)
    source_colors = [source_colors[i % len(source_colors)] for i in range(num_sources)]

    # Target-node colours
    target_colors = [target_color] * len(target_categories)
    node_colors = source_colors + target_colors

    # 4. Build the link list
    links = []
    for source_idx, source in enumerate(source_categories):
        source_color = source_colors[source_idx]
        r = int(source_color[1:3], 16)
        g = int(source_color[3:5], 16)
        b = int(source_color[5:7], 16)
        link_color = f'rgba({r}, {g}, {b}, 0.3)'

        for target in target_categories:
            value = count_matrix.loc[source, target]
            if value > 0:
                links.append({
                    'source': node_indices[source],
                    'target': node_indices[f"{target} "], # Matches target_categories_renamed on line 32
                    'value': value,
                    'color': link_color
                })

    # 5. Create the Sankey diagram
    sankey_data = go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_categories,
            color=node_colors
        ),
        link=dict(
            source=[link['source'] for link in links],
            target=[link['target'] for link in links],
            value=[link['value'] for link in links],
            color=[link['color'] for link in links]
        )
    )

    fig = go.Figure(data=[sankey_data])
    fig.add_annotation(
        x=0, y=1.06, # Top left
        xref="paper", yref="paper",
        text=f"<b>Annotated ({source_col})</b>",
        showarrow=False,
        font=dict(size=14, color="black"),
        align="left"
    )
    fig.add_annotation(
        x=1, y=1.06, # Top right
        xref="paper", yref="paper",
        text=f"<b>Predicted ({target_col})</b>",
        showarrow=False,
        font=dict(size=14, color="black"),
        align="right"
    )
    fig.update_layout(
        title_text=title,
        font_size=10,
        width=width,
        height=height
    )
    

    # 6. Output handling
    if interactive:
        fig.show()
    else:
        img_bytes = pio.to_image(fig, format="png", width=width, height=height, scale=2)
        from IPython.display import Image, display
        display(Image(img_bytes))  # Render statically in the notebook
        if output_file is not None:
            pio.write_image(fig, output_file, width=width, height=height, scale=2)


In [ ]:
def build_sankey_from_layers(df, layer_cols, title, out_path, height=1200, width=1200):
    import plotly.graph_objects as go
    import plotly.io as pio
    """
    df: DataFrame containing the layer_cols (one row per cell)
    layer_cols: their order is the flow order
    """
    # 1) Filter: drop rows where any layer is empty
    df = df[layer_cols].copy()
    df = df.dropna(how="any")



    # 3) Count each flow segment
    flows = []
    for i in range(len(layer_cols) - 1):
        src = layer_cols[i]
        tgt = layer_cols[i + 1]
        g = df.groupby([src, tgt]).size().reset_index(name="value")
        g["src_layer"] = src
        g["tgt_layer"] = tgt
        flows.append(g)

    flows = pd.concat(flows, ignore_index=True)

    # 4) Build nodes (unique internally by (layer, value); labels show only value)
    #    Note: a set cannot be used as an indexer here (seen before), so use pd.unique + concat throughout
    src_node_keys = flows.apply(lambda r: (r["src_layer"], r[r["src_layer"]]), axis=1)
    tgt_node_keys = flows.apply(lambda r: (r["tgt_layer"], r[r["tgt_layer"]]), axis=1)
    node_keys = pd.unique(pd.concat([src_node_keys, tgt_node_keys], ignore_index=True))

    node_key_to_id = {k: i for i, k in enumerate(node_keys)}
    node_labels = [k[1] for k in node_keys]  # Show only the cell-type name

    # 5) Build the Sankey links
    sources, targets, values = [], [], []
    for _, r in flows.iterrows():
        src_key = (r["src_layer"], r[r["src_layer"]])
        tgt_key = (r["tgt_layer"], r[r["tgt_layer"]])
        sources.append(node_key_to_id[src_key])
        targets.append(node_key_to_id[tgt_key])
        values.append(int(r["value"]))

    # 6) Plot
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            label=node_labels,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
        )
    ))

    fig.update_layout(
        title_text=title,
        height=height,
        width=width,
        font_size=12,
    )
    img_bytes = pio.to_image(fig, format="png", width=width, height=height, scale=2)
    from IPython.display import Image, display
    display(Image(img_bytes))
    
    # SVG needs kaleido; if it is not installed, run `pip install -U kaleido` first
    fig.write_image(out_path, scale=3)
    return fig


In [ ]:
# def knn_predict(adata,train_species,test_species,cell_column='std_cell_type',k=15,sub_sample=False,num_lim=350):
#     from sklearn.neighbors import KNeighborsClassifier
#     from sklearn.preprocessing import LabelEncoder
#     adata_train=adata[adata.obs['sp']==train_species,:]
#     adata_test=adata[adata.obs['sp']==test_species,:]
#     if sub_sample:
#         counts = adata_train.obs[cell_column].value_counts()
#         keep_types = counts[counts >= num_lim].index
#         adata_train = adata_train[adata_train.obs[cell_column].isin(keep_types), :].copy()
#         counts = adata_test.obs[cell_column].value_counts()
#         keep_types = counts[counts >= num_lim].index
#         adata_test = adata_test[adata_test.obs[cell_column].isin(keep_types), :].copy()
        
#         shared_cells=set(adata_train.obs[cell_column])&set(adata_test.obs[cell_column])
#         print(shared_cells)
#         adata_test=adata_test[adata_test.obs[cell_column].isin(shared_cells),:]
#         adata_train=adata_train[adata_train.obs[cell_column].isin(shared_cells),:]
#     if len(adata_test)==0 or len(adata_train)==0:
#         return None
#     knn = KNeighborsClassifier(n_neighbors=k)
#     le = LabelEncoder()
#     le.fit(adata_train.obs[cell_column])
#     y_train_encoded = le.transform(adata_train.obs[cell_column])
#     knn.fit(adata_train.obsm['X_HENOSIS'], y_train_encoded)
#     y_pred_encoded = knn.predict(adata_test.obsm['X_HENOSIS'])
#     '''knn.fit(adata_train.X, y_train_encoded)
#     y_pred_encoded = knn.predict(adata_test.X)'''
#     y_pred = le.inverse_transform(y_pred_encoded)
    
#     adata_test.obs['pred_label']=y_pred
#     return adata_test

def knn_predict(
    adata,
    train_species,
    test_species,
    cell_column,
    use_rep="X_HENOSIS",
    k=15,
    metric="cosine",
    weights="distance",
    sub_sample=False,
    num_lim=350,
    only_shared_celltypes=True,
    normalize=False,
    return_proba=True,
):
    import numpy as np
    import pandas as pd
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.preprocessing import LabelEncoder, normalize as sk_normalize

    # =========================
    # 1. Check
    # =========================
    if "sp" not in adata.obs.columns:
        raise ValueError("no 'sp' column in adata.obs")

    if cell_column not in adata.obs.columns:
        raise ValueError(f"no '{cell_column}' column in adata.obs")

    if use_rep not in adata.obsm.keys():
        raise ValueError(f"no '{use_rep}' in adata.obsm")

    # =========================
    # 2. Split by species
    # =========================
    adata_train = adata[adata.obs["sp"] == train_species, :].copy()
    adata_test = adata[adata.obs["sp"] == test_species, :].copy()

    if adata_train.n_obs == 0:
        print(f"No training cells found for species: {train_species}")
        return None

    if adata_test.n_obs == 0:
        print(f"No test cells found for species: {test_species}")
        return None

    # =========================
    # 3. Drop missing labels
    # =========================
    adata_train = adata_train[
        adata_train.obs[cell_column].notna()
    ].copy()

    adata_test = adata_test[
        adata_test.obs[cell_column].notna()
    ].copy()

    adata_train.obs[cell_column] = adata_train.obs[cell_column].astype(str)
    adata_test.obs[cell_column] = adata_test.obs[cell_column].astype(str)

    # =========================
    # 4. Optional: drop low-frequency cell types
    # =========================
    if sub_sample:
        train_counts = adata_train.obs[cell_column].value_counts()
        train_keep_types = train_counts[train_counts >= num_lim].index

        test_counts = adata_test.obs[cell_column].value_counts()
        test_keep_types = test_counts[test_counts >= num_lim].index

        adata_train = adata_train[
            adata_train.obs[cell_column].isin(train_keep_types)
        ].copy()

        adata_test = adata_test[
            adata_test.obs[cell_column].isin(test_keep_types)
        ].copy()

    # =========================
    # 5. Optional: keep only cell types shared by train and test
    # =========================
    if only_shared_celltypes:
        shared_celltypes = (
            set(adata_train.obs[cell_column].astype(str))
            & set(adata_test.obs[cell_column].astype(str))
        )

        shared_celltypes = sorted(shared_celltypes)

        print("Shared cell types:")
        print(shared_celltypes)

        adata_train = adata_train[
            adata_train.obs[cell_column].isin(shared_celltypes)
        ].copy()

        adata_test = adata_test[
            adata_test.obs[cell_column].isin(shared_celltypes)
        ].copy()

    if adata_train.n_obs == 0 or adata_test.n_obs == 0:
        print("No cells left after filtering.")
        return None

    # =========================
    # 6. Extract the embedding
    # =========================
    X_train = adata_train.obsm[use_rep]
    X_test = adata_test.obsm[use_rep]

    X_train = np.asarray(X_train)
    X_test = np.asarray(X_test)

    # For cosine distance, L2 normalisation is usually more stable
    if normalize or metric == "cosine":
        X_train = sk_normalize(X_train, norm="l2", axis=1)
        X_test = sk_normalize(X_test, norm="l2", axis=1)

    # =========================
    # 7. Encode the labels
    # =========================
    le = LabelEncoder()
    y_train = adata_train.obs[cell_column].astype(str).values
    le.fit(y_train)

    y_train_encoded = le.transform(y_train)

    # =========================
    # 8. Train KNN
    # =========================
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric=metric,
        weights=weights,
        n_jobs=-1,
    )

    knn.fit(X_train, y_train_encoded)

    # =========================
    # 9. Predict
    # =========================
    y_pred_encoded = knn.predict(X_test)
    y_pred = le.inverse_transform(y_pred_encoded)

    adata_test.obs["pred_label"] = y_pred

    if return_proba:
        proba = knn.predict_proba(X_test)
        max_proba = proba.max(axis=1)

        adata_test.obs["pred_confidence"] = max_proba

        # The per-class probabilities can also be saved as a DataFrame
        proba_df = pd.DataFrame(
            proba,
            index=adata_test.obs_names,
            columns=[f"proba_{x}" for x in le.classes_],
        )

        for col in proba_df.columns:
            adata_test.obs[col] = proba_df[col].values

    return adata_test


In [ ]:
organ='spaa'
figure='figure6'
save_dir=f"./umap_results/{figure}/{organ}"
os.makedirs(save_dir, exist_ok=True)


In [ ]:
adata = sc.read_h5ad(
    DATA_ROOT / "Figure4/xenium/brain_integrated.h5ad"
)


In [ ]:
adata.obs['Celltype'].unique()


In [ ]:
import pandas as pd

# 1. Force to string to avoid stale category levels
species_col = 'sp'  # 👈 Check and change your species column name
adata.obs['Celltype'] = adata.obs['Celltype'].astype(str)
adata.obs[species_col] = adata.obs[species_col].astype(str)

# 2. Build the cross-tabulation
cell_counts_matrix = pd.crosstab(
    index=adata.obs[species_col], 
    columns=adata.obs['Celltype']
)

# Display the full cross-tabulation
print(cell_counts_matrix)


In [ ]:
adata


In [ ]:
import pandas as pd
import os

train_species_list = ['homo_sapiens', 'Mus_musculus']
true_col = 'Celltype'            # Column holding the true labels
pred_col = 'pred_label'                # Column holding the predicted labels
species_col = 'sp'

# ==================== 1. Cell-type filtering ====================
# Convert the label columns to string first so nan is handled uniformly
adata.obs[true_col] = adata.obs[true_col].astype(str)

# Exclude 'Vascular/Stromal' and nan in its various spellings
exclude_list = ['Vascular/Stromal', 'nan', 'NaN']
adata = adata[~adata.obs[true_col].isin(exclude_list)].copy()

print(f"Cell types present after filtering:\n{adata.obs[true_col].unique()}\n")
# ========================================================

# Re-extract the species list after filtering
species_list = list(dict.fromkeys(adata.obs[species_col]))

save_dir = './figure_result'
organ = 'brain'

for train_species in train_species_list:
    for species in species_list:
        if species == train_species:
            continue
        print('train_species:', train_species)
        print('test_species:', species)
        
        # Create the output directory
        train_save_dir = f'{save_dir}/{train_species}_reference'
        os.makedirs(train_save_dir, exist_ok=True)
        
        # # 2. Subsampled prediction (kept only as a fallback; no plots ❌)
        # adata_test_sub = knn_predict(adata, train_species=train_species, test_species=species, sub_sample=True, cell_column=true_col, num_lim=0)
        
        # 3. Full-data prediction (✅ the CM and Sankey plots are based on this full result)
        # adata_test_full = knn_predict(adata, train_species=train_species, test_species=species,cell_column=true_col,  sub_sample=False, num_lim=0)
        adata_test_full = knn_predict(
            adata, train_species=train_species, test_species=species,cell_column=true_col, 
            use_rep="X_HENOSIS",
            k=101,
            metric="cosine",
            weights="distance",
            sub_sample=False,
            num_lim=350,
            only_shared_celltypes=True,
        )
                
        if adata_test_full is not None:
            # ==== [Added: draw the CM from the full data] ====
            df_cm = adata_test_full.obs[[true_col, pred_col]].copy()
            df_cm[true_col] = df_cm[true_col].astype(str)
            df_cm[pred_col] = df_cm[pred_col].astype(str)
            
            plot_cm_from_df(
                df=df_cm, 
                true_col=true_col, 
                pred_col=pred_col, 
                save_name=f'{train_save_dir}/{organ}_{train_species}_{species}_cm_plot', 
                species=None,   
                threshold=0.10
            )
            
            # # ==== [Kept: draw the three-layer Sankey from the full data] ====
            # build_sankey_from_layers(
            #     df=adata_test_full.obs[[true_col, 'leiden', 'pred_label']],
            #     layer_cols=[f'{true_col}', 'leiden', 'pred_label'],
            #     title=f"{true_col} ← leiden → pred_label",
            #     out_path=f'{train_save_dir}/{organ}_{train_species}_{species}_leiden_sankey.svg',
            #     height=1200,
            #     width=1800
            # )


In [ ]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


# ============================================================
# 0. Parameters
# ============================================================

train_species_list = ["homo_sapiens", "Mus_musculus"]

true_col = "Celltype"
pred_col = "pred_label"
species_col = "sp"

organ = "brain"
save_dir = "./figure_result"

k_list = [ 51, 75, 101, 200, 300]

use_rep = "X_HENOSIS"
metric = "cosine"
weights = "distance"

exclude_list = ["Vascular/Stromal", "nan", "NaN"]


# ============================================================
# 1. Cell-type filtering
# ============================================================

adata.obs[true_col] = adata.obs[true_col].astype(str)

adata = adata[
    ~adata.obs[true_col].isin(exclude_list)
].copy()

print(f"Cell types present after filtering:")
print(adata.obs[true_col].unique())
print()

species_list = list(dict.fromkeys(adata.obs[species_col].astype(str)))

print("Species list:")
print(species_list)


# ============================================================
# 2. Evaluation function
# ============================================================

def evaluate_prediction_df(df, true_col, pred_col):
    """
    Input: a DataFrame with true_col and pred_col.
    Returns accuracy, macro_f1 and weighted_f1.
    """
    y_true = df[true_col].astype(str).values
    y_pred = df[pred_col].astype(str).values

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    return acc, macro_f1, weighted_f1


# ============================================================
# 3. Loop over different k values
# ============================================================

summary_records = []

for train_species in train_species_list:
    for species in species_list:

        if species == train_species:
            continue

        print("\n" + "=" * 80)
        print("train_species:", train_species)
        print("test_species:", species)

        for k in k_list:
            print("\n" + "-" * 60)
            print(f"Running KNN: k={k}")

            # Create a separate directory per training species and k
            train_save_dir = os.path.join(
                save_dir,
                f"{train_species}_reference",
                f"k_{k}"
            )
            os.makedirs(train_save_dir, exist_ok=True)

            # ====================================================
            # 3.1 KNN prediction
            # ====================================================
            adata_test_full = knn_predict(
                adata,
                train_species=train_species,
                test_species=species,
                cell_column=true_col,
                use_rep=use_rep,
                k=k,
                metric=metric,
                weights=weights,
                sub_sample=False,
                num_lim=350,
                only_shared_celltypes=True,
            )

            if adata_test_full is None:
                print(f"Skip: {train_species} -> {species}, k={k}, no result.")

                summary_records.append({
                    "organ": organ,
                    "train_species": train_species,
                    "test_species": species,
                    "k": k,
                    "use_rep": use_rep,
                    "metric": metric,
                    "weights": weights,
                    "n_cells": 0,
                    "n_true_celltypes": 0,
                    "accuracy": None,
                    "macro_f1": None,
                    "weighted_f1": None,
                    "status": "no_result",
                })

                continue

            # ====================================================
            # 3.2 Extract the predictions
            # ====================================================
            df_pred = adata_test_full.obs[[true_col, pred_col]].copy()
            df_pred[true_col] = df_pred[true_col].astype(str)
            df_pred[pred_col] = df_pred[pred_col].astype(str)

            n_cells = df_pred.shape[0]
            n_true_celltypes = df_pred[true_col].nunique()

            acc, macro_f1, weighted_f1 = evaluate_prediction_df(
                df=df_pred,
                true_col=true_col,
                pred_col=pred_col,
            )

            print(f"n_cells: {n_cells}")
            print(f"n_true_celltypes: {n_true_celltypes}")
            print(f"accuracy: {acc:.4f}")
            print(f"macro_f1: {macro_f1:.4f}")
            print(f"weighted_f1: {weighted_f1:.4f}")

            summary_records.append({
                "organ": organ,
                "train_species": train_species,
                "test_species": species,
                "k": k,
                "use_rep": use_rep,
                "metric": metric,
                "weights": weights,
                "n_cells": n_cells,
                "n_true_celltypes": n_true_celltypes,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
                "status": "success",
            })

            # ====================================================
            # 3.3 Save the prediction table
            # ====================================================
            pred_csv = os.path.join(
                train_save_dir,
                f"{organ}_{train_species}_{species}_k{k}_prediction.csv"
            )

            adata_test_full.obs[
                [true_col, pred_col]
            ].to_csv(pred_csv)

            print(f"Saved prediction CSV: {pred_csv}")

            # ====================================================
            # 3.4 Draw the CM
            # ====================================================
            cm_save_name = os.path.join(
                train_save_dir,
                f"{organ}_{train_species}_{species}_k{k}_cm_plot"
            )

            plot_cm_from_df(
                df=df_pred,
                true_col=true_col,
                pred_col=pred_col,
                save_name=cm_save_name,
                species=None,
                threshold=0.10,
            )

            # ====================================================
            # 3.5 Draw the three-layer Sankey
            # ====================================================
            # sankey_df = adata_test_full.obs[
            #     [true_col, "leiden", pred_col]
            # ].copy()

            # sankey_out = os.path.join(
            #     train_save_dir,
            #     f"{organ}_{train_species}_{species}_k{k}_leiden_sankey.svg"
            # )

            # build_sankey_from_layers(
            #     df=sankey_df,
            #     layer_cols=[true_col, "leiden", pred_col],
            #     title=f"{true_col} ← leiden → {pred_col} | k={k}",
            #     out_path=sankey_out,
            #     height=1200,
            #     width=1800,
            # )


# ============================================================
# 4. Save the k comparison
# ============================================================

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    save_dir,
    f"{organ}_knn_k_summary.csv"
)

summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 80)
print("Finished all KNN runs.")
print("Summary saved to:")
print(summary_path)

print("\nTop results by macro_f1:")
print(
    summary_df
    .dropna(subset=["macro_f1"])
    .sort_values("macro_f1", ascending=False)
    .head(20)
)


In [ ]:
import pandas as pd
import os
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder

def svm_predict(adata, train_species, test_species, cell_column='Celltype', sub_sample=False, num_lim=0):
    """
    Cross-species cell-type prediction using a Linear SVM instead of KNN.
    """
    # 1. Split into training and test sets
    adata_train = adata[adata.obs['sp'] == train_species, :].copy()
    adata_test = adata[adata.obs['sp'] == test_species, :].copy()
    
    # 2. Optional subsampling (keeps the original filtering logic)
    if sub_sample:
        counts = adata_train.obs[cell_column].value_counts()
        keep_types = counts[counts >= num_lim].index
        adata_train = adata_train[adata_train.obs[cell_column].isin(keep_types), :]
        
        counts = adata_test.obs[cell_column].value_counts()
        keep_types = counts[counts >= num_lim].index
        adata_test = adata_test[adata_test.obs[cell_column].isin(keep_types), :]
        
        shared_cells = set(adata_train.obs[cell_column]) & set(adata_test.obs[cell_column])
        adata_test = adata_test[adata_test.obs[cell_column].isin(shared_cells), :]
        adata_train = adata_train[adata_train.obs[cell_column].isin(shared_cells), :]
        
    if len(adata_test) == 0 or len(adata_train) == 0:
        return None
        
    # 3. Encode the labels
    le = LabelEncoder()
    # Force to string to avoid the category bug
    train_labels = adata_train.obs[cell_column].astype(str)
    le.fit(train_labels)
    y_train_encoded = le.transform(train_labels)
    
    # 4. Train the SVM (LinearSVC performs very well on high-dimensional single-cell features)
    clf = LinearSVC(max_iter=10000, dual=False, random_state=42)
    clf.fit(adata_train.obsm['X_HENOSIS'], y_train_encoded)
    
    # 5. Predict
    y_pred_encoded = clf.predict(adata_test.obsm['X_HENOSIS'])
    y_pred = le.inverse_transform(y_pred_encoded)
    
    adata_test.obs['pred_label'] = y_pred
    return adata_test




# ==================== Main workflow ====================

train_species_list = ['homo_sapiens', 'Mus_musculus']
true_col = 'Celltype'            # Column holding the true labels
pred_col = 'pred_label'                # Column holding the predicted labels
species_col = 'sp'

# 1. Global cell-type pre-filtering (remove noise and empty values)
adata.obs[true_col] = adata.obs[true_col].astype(str)
exclude_list = ['Vascular/Stromal', 'nan', 'NaN']
adata = adata[~adata.obs[true_col].isin(exclude_list)].copy()

print(f"True cell types present after filtering:\n{list(adata.obs[true_col].unique())}\n")

# Re-extract the species list after filtering
species_list = list(dict.fromkeys(adata.obs[species_col]))

save_dir = './figure_result'
organ = 'brain'

for train_species in train_species_list:
    for species in species_list:
        if species == train_species:
            continue
        print(f"\n>>> Running prediction: train species [{train_species}] --> test species [{species}]")
        
        # Create the output directory
        train_save_dir = f'{save_dir}/{train_species}_reference'
        os.makedirs(train_save_dir, exist_ok=True)
        
        # 2. Full-data prediction (with the improved SVM)
        adata_test_full = svm_predict(adata, train_species=train_species, test_species=species, cell_column=true_col, sub_sample=False)
        
        if adata_test_full is not None:
            # ==== Fix the alignment: prevent axis-label mismatch in the plotting function ====
            df_cm = adata_test_full.obs[[true_col, pred_col]].copy()
            
            # Merge the full cell-type range of both species
            all_categories = sorted(list(set(df_cm[true_col].unique()) | set(df_cm[pred_col].unique())))
            
            # Use one consistent categorical type so column X of the CM is exactly cell type X
            df_cm[true_col] = pd.Categorical(df_cm[true_col], categories=all_categories)
            df_cm[pred_col] = pd.Categorical(df_cm[pred_col], categories=all_categories)
            
            # ==== Plot: CM ====
            plot_cm_from_df(
                df=df_cm, 
                true_col=true_col, 
                pred_col=pred_col, 
                save_name=f'{train_save_dir}/{organ}_{train_species}_{species}_cm_plot', 
                species=None,   
                threshold=0.10
            )
            
            # ==== Plot: three-layer Sankey ====
            # build_sankey_from_layers(
            #     df=adata_test_full.obs[[true_col, 'leiden', 'pred_label']],
            #     layer_cols=[true_col, 'leiden', 'pred_label'],
            #     title=f"{true_col} ← leiden → pred_label",
            #     out_path=f'{train_save_dir}/{organ}_{train_species}_{species}_leiden_sankey.svg',
            #     height=1200,
            #     width=1800
            # )


In [ ]:
import numpy as np
import pandas as pd
import os
import scanpy as sc

def correlation_predict(adata, train_species, test_species, cell_column='Celltype'):
    """
    Cell-type prediction from the correlation between cross-species ortholog expression profiles (centroid correlation).
    """
    # 1. Extract the training and test sets
    adata_train = adata[adata.obs['sp'] == train_species, :].copy()
    adata_test = adata[adata.obs['sp'] == test_species, :].copy()
    
    if len(adata_test) == 0 or len(adata_train) == 0:
        print(f"⚠️ Warning: species {train_species} or {test_species} has no data; skipping prediction.")
        return None
        
    # 2. Find genes shared by both species (orthologs)
    shared_genes = list(set(adata_train.var_names) & set(adata_test.var_names))
    if len(shared_genes) < 20:
        print("❌ Error: too few shared genes between the two species; check that var_names use a consistent case convention!")
        return None
    
    # 3. Slice out the shared genes
    adata_train_sub = adata_train[:, shared_genes]
    adata_test_sub = adata_test[:, shared_genes]
    
    # 4. Densify the sparse matrix for the numerical computation
    train_expr = adata_train_sub.X.toarray() if hasattr(adata_train_sub.X, "toarray") else adata_train_sub.X
    test_expr = adata_test_sub.X.toarray() if hasattr(adata_test_sub.X, "toarray") else adata_test_sub.X
    
    # 5. Build the training DataFrame and compute the mean expression profile (centroid) per cell type
    train_df = pd.DataFrame(train_expr, columns=shared_genes)
    train_df['cell_type'] = adata_train.obs[cell_column].astype(str).values
    centroids = train_df.groupby('cell_type').mean() 
    
    # 6. Compute the Pearson correlation of each test cell with each training centroid
    # To avoid NaNs when a gene is zero in every cell (zero denominator), perturb slightly before np.corrcoef or use np.nan_to_num
    print("Computing Pearson correlations between cell expression profiles and reference centroids...")
    corr_matrix = np.corrcoef(test_expr, centroids.values)[:len(test_expr), len(test_expr):]
    corr_matrix = np.nan_to_num(corr_matrix, nan=-1.0) # Assign a very low correlation where it cannot be computed (dead cells / all-zero profiles)
    
    # 7. Predict the cell type with the highest correlation
    best_match_idx = np.argmax(corr_matrix, axis=1)
    pred_labels = centroids.index[best_match_idx]
    
    adata_test.obs['pred_label'] = pred_labels
    return adata_test

# ==================== Approach 1: main workflow ====================

train_species_list = ['homo_sapiens', 'Mus_musculus']
true_col = 'Celltype'            # Column holding the true labels
pred_col = 'pred_label'                # Column holding the predicted labels
species_col = 'sp'

# Global cell-type pre-filtering
adata.obs[true_col] = adata.obs[true_col].astype(str)
exclude_list = ['Vascular/Stromal', 'nan', 'NaN']
adata = adata[~adata.obs[true_col].isin(exclude_list)].copy()

print(f"True cell types present after filtering:\n{list(adata.obs[true_col].unique())}\n")

species_list = list(dict.fromkeys(adata.obs[species_col]))
save_dir = './figure_result_correlation'
organ = 'brain'

for train_species in train_species_list:
    for species in species_list:
        if species == train_species:
            continue
        print(f"\n>>> [centroid correlation] train: {train_species} --> test: {species}")
        
        train_save_dir = f'{save_dir}/{train_species}_reference'
        os.makedirs(train_save_dir, exist_ok=True)
        
        # Call the correlation-based predictor
        adata_test_full = correlation_predict(adata, train_species=train_species, test_species=species, cell_column=true_col)
        
        if adata_test_full is not None:
            # ==== Fix the alignment to prevent label mismatch in the plotting function ====
            df_cm = adata_test_full.obs[[true_col, pred_col]].copy()
            all_categories = sorted(list(set(df_cm[true_col].unique()) | set(df_cm[pred_col].unique())))
            df_cm[true_col] = pd.Categorical(df_cm[true_col], categories=all_categories)
            df_cm[pred_col] = pd.Categorical(df_cm[pred_col], categories=all_categories)
            
            # Draw the CM
            plot_cm_from_df(
                df=df_cm, 
                true_col=true_col, 
                pred_col=pred_col, 
                save_name=f'{train_save_dir}/{organ}_{train_species}_{species}_cm_plot', 
                species=None,   
                threshold=0.10
            )
            
            # # Draw the three-layer Sankey
            # build_sankey_from_layers(
            #     df=adata_test_full.obs[[true_col, 'leiden', 'pred_label']],
            #     layer_cols=[true_col, 'leiden', 'pred_label'],
            #     title=f"{true_col} ← leiden → pred_label",
            #     out_path=f'{train_save_dir}/{organ}_{train_species}_{species}_leiden_sankey.svg',
            #     height=1200,
            #     width=1800
            # )


In [ ]:
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

def rf_predict(adata, train_species, test_species, cell_column='Celltype', n_estimators=100):
    """
    Cross-species cell-type prediction with a random forest in Harmony space.
    """
    # 1. Split into training and test sets
    adata_train = adata[adata.obs['sp'] == train_species, :].copy()
    adata_test = adata[adata.obs['sp'] == test_species, :].copy()
    
    if len(adata_test) == 0 or len(adata_train) == 0:
        print(f"⚠️ Warning: species {train_species} or {test_species} has no data; skipping prediction.")
        return None
        
    # 2. Encode labels as integers
    le = LabelEncoder()
    train_labels = adata_train.obs[cell_column].astype(str)
    le.fit(train_labels)
    y_train_encoded = le.transform(train_labels)
    
    # 3. Initialise and train the random-forest model
    # n_jobs=-1 uses every CPU core in parallel
    print(f"Training the random-forest model (number of trees: {n_estimators})...")
    clf = RandomForestClassifier(n_estimators=n_estimators, n_jobs=-1, random_state=42)
    clf.fit(adata_train.obsm['X_HENOSIS'], y_train_encoded)
    
    # 4. Predict on the test set
    print("Classifying the target-species cells...")
    y_pred_encoded = clf.predict(adata_test.obsm['X_HENOSIS'])
    
    # 5. Convert the integer labels back to strings
    y_pred = le.inverse_transform(y_pred_encoded)
    
    adata_test.obs['pred_label'] = y_pred
    return adata_test

# ==================== Approach 2: main workflow ====================

train_species_list = ['homo_sapiens', 'Mus_musculus']
true_col = 'Celltype'            
pred_col = 'pred_label'                
species_col = 'sp'

# Global cell-type pre-filtering
adata.obs[true_col] = adata.obs[true_col].astype(str)
exclude_list = ['Vascular/Stromal', 'nan', 'NaN']
adata = adata[~adata.obs[true_col].isin(exclude_list)].copy()

species_list = list(dict.fromkeys(adata.obs[species_col]))
save_dir = './figure_result_rf'
organ = 'brain'

for train_species in train_species_list:
    for species in species_list:
        if species == train_species:
            continue
        print(f"\n>>> [random forest] train: {train_species} --> test: {species}")
        
        train_save_dir = f'{save_dir}/{train_species}_reference'
        os.makedirs(train_save_dir, exist_ok=True)
        
        # Call the random-forest predictor (150 trees for better cross-species generalisation)
        adata_test_full = rf_predict(adata, train_species=train_species, test_species=species, cell_column=true_col, n_estimators=150)
        
        if adata_test_full is not None:
            # ==== Fix the alignment to prevent label mismatch in the plotting function ====
            df_cm = adata_test_full.obs[[true_col, pred_col]].copy()
            all_categories = sorted(list(set(df_cm[true_col].unique()) | set(df_cm[pred_col].unique())))
            df_cm[true_col] = pd.Categorical(df_cm[true_col], categories=all_categories)
            df_cm[pred_col] = pd.Categorical(df_cm[pred_col], categories=all_categories)
            
            # Draw the CM
            plot_cm_from_df(
                df=df_cm, 
                true_col=true_col, 
                pred_col=pred_col, 
                save_name=f'{train_save_dir}/{organ}_{train_species}_{species}_cm_plot', 
                species=None,   
                threshold=0.10
            )
            
            # Draw the three-layer Sankey
            # build_sankey_from_layers(
            #     df=adata_test_full.obs[[true_col, 'leiden', 'pred_label']],
            #     layer_cols=[true_col, 'leiden', 'pred_label'],
            #     title=f"{true_col} ← leiden → pred_label",
            #     out_path=f'{train_save_dir}/{organ}_{train_species}_{species}_leiden_sankey.svg',
            #     height=1200,
            #     width=1800
            # )


In [ ]:
adata.obs['std_cell_type']=adata.obs['Celltype']


In [ ]:
adata.obs['std_cell_type']=adata.obs['Celltype']


In [ ]:
train_species_list=['homo_sapiens','Mus_musculus'
                   ]
species_list=list(dict.fromkeys(adata.obs['sp']))
true_col = 'Celltype'            # Column holding the true labels
pred_col = 'pred_label'  # Column holding the predicted labels
species_col = 'sp'


In [ ]:
save_dir = './figure_result'
organ = 'brain'
for train_species in train_species_list:
    for species in species_list:
        if species==train_species:
            continue
        print('train_species:',train_species)
        print('test_species:',species)
        adata_test=knn_predict(adata,train_species=train_species,test_species=species,sub_sample=True,cell_column=true_col,num_lim=0)
        if adata_test is not None:
            df_cm = adata_test.obs[[true_col, pred_col]].copy()
            df_cm[true_col] = df_cm[true_col].astype(str)
            df_cm[pred_col] = df_cm[pred_col].astype(str)
            train_save_dir=f'{save_dir}/{train_species}_reference'
            os.makedirs(train_save_dir,exist_ok=True)
            plot_cm_from_df(
                df=df_cm, 
                true_col=true_col, 
                pred_col=pred_col, 
                save_name=f'{train_save_dir}/{organ}_{train_species}_{species}_cm_plot', 
                species=None,   # Passed to the plotting function for naming
                threshold=0.10
            )
            # plot_sankey_from_adata(adata_test,true_col,'pred_label',output_file=f'{train_save_dir}/{organ}_{train_species}_{species}_sankey.svg')
        adata_test=knn_predict(adata,train_species=train_species,test_species=species,sub_sample=False,num_lim=0)
        build_sankey_from_layers(
            df=adata_test.obs[[true_col,'leiden','pred_label']],
            layer_cols=[f'{true_col}','leiden','pred_label'],
            title=f"{true_col} ← leiden → pred_label",
            out_path=f'{train_save_dir}/{organ}_{train_species}_{species}_leiden_sankey.svg',
            height=1200,
            width=1800
        )
